# 🧹 Notebook 02 — Data Preprocessing
**Goal:** Clean, encode, and split the raw dataset into train/val/test sets.

---

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os, warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

✅ Libraries loaded


## 1️⃣ Load Raw Data

In [2]:
df = pd.read_csv('../data/raw/churn_raw.csv')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (7043, 17)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,ID-00000,Male,0,Yes,No,42,Yes,No,Fiber optic,No internet service,Yes,Month-to-month,Yes,Electronic check,76.12,1399.70,Yes
1,ID-00001,Female,1,Yes,Yes,68,Yes,No phone service,No,Yes,No internet service,Month-to-month,Yes,Credit card (automatic),38.32,7210.27,Yes
2,ID-00002,Male,0,No,Yes,62,Yes,Yes,No,No,No internet service,Month-to-month,No,Mailed check,21.06,2921.38,No


## 2️⃣ Drop Irrelevant Columns

In [3]:
df.drop(columns=['customerID'], inplace=True)
print(f'After drop: {df.shape}')

After drop: (7043, 16)


## 3️⃣ Handle Missing Values

In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
print(f'Missing values: {df.isnull().sum().sum()}')

Missing values: 0


## 4️⃣ Encode Target

In [5]:
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print(df['Churn'].value_counts())

Churn
0    5082
1    1961
Name: count, dtype: int64


## 5️⃣ Encode Features

In [6]:
# Binary encoding
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']
for col in binary_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

# One-Hot encoding
multi_cols = ['MultipleLines','InternetService','OnlineSecurity','TechSupport','Contract','PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)
print(f'After encoding: {df.shape}')

After encoding: (7043, 23)


## 6️⃣ Train / Val / Test Split (70/15/15)

In [7]:
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

Train: (4930, 22) | Val: (1056, 22) | Test: (1057, 22)


## 7️⃣ Save Splits

In [8]:
os.makedirs('../data/processed', exist_ok=True)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
df.to_csv('../data/processed/churn_processed.csv', index=False)
print('✅ All splits saved!')
print(os.listdir('../data/processed/'))

✅ All splits saved!
['01_categorical_churn.png', '01_churn_distribution.png', '01_numerical_features.png', 'churn_processed.csv', 'X_test.csv', 'X_train.csv', 'X_val.csv', 'y_test.csv', 'y_train.csv', 'y_val.csv']


## ✅ Summary

Splits: 70/15/15 stratified | Binary + OHE encoding done

➡️ Next: `03_Feature_Engineering.ipynb`